# 04 — Questions hors périmètre et décisions d’abstention

Un assistant doit pouvoir refuser une question lorsque la base ne fournit pas de
contexte suffisamment pertinent. Ce notebook examine une première règle de décision
fondée sur le meilleur score retourné par chaque méthode de recherche.

**Objectif :** observer l’effet des seuils et mesurer le compromis entre faux refus
et fausses acceptations.

Le parcours présente les scores des questions, applique un seuil propre à chaque
méthode, mesure les erreurs de décision, puis explore plusieurs seuils. Les valeurs
étudiées sont illustratives : elles ne constituent pas une calibration validée.


In [1]:
import logging

# Message macOS concernant une redirection multiprocessus inutilisée ici.
# Les autres avertissements et erreurs restent visibles.
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").addFilter(
    lambda record: record.getMessage() != "NOTE: Redirects are currently not supported in MacOs."
)

In [2]:
from IPython.display import Markdown, display

from contextiq.evaluation.abstention_metrics import AbstentionEvaluator
from contextiq.ingestion.loader import DataLoader
from contextiq.retrieval.bm25 import BM25Retriever
from contextiq.retrieval.embeddings import EmbeddingRetriever
from contextiq.retrieval.tfidf import TfidfRetriever
from contextiq.safety.abstention import AbstentionPolicy
from contextiq.settings import Settings

settings = Settings.load()
dataset = DataLoader().load()
retrievers = {
    "TF-IDF": TfidfRetriever(dataset.documents),
    "BM25": BM25Retriever(dataset.documents, **settings.retrieval.bm25.model_dump()),
    "Embeddings": EmbeddingRetriever(
        dataset.documents, **settings.retrieval.embeddings.model_dump()
    ),
}

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 1. Examiner les scores des questions

Les meilleurs scores sont présentés par méthode, en distinguant les questions avec
source des questions hors périmètre. Cette lecture permet d’observer leur séparation
éventuelle. Les trois échelles ne sont pas directement comparables et le jeu ne
contient qu’une question hors périmètre.


In [3]:
for name, retriever in retrievers.items():
    print(f"\n{name}")
    for item in dataset.questions:
        results = retriever.retrieve(item.q, top_k=settings.retrieval.top_k)
        best_score = max((result.score for result in results), default=0.0)
        scope = "avec source" if item.gold_docs else "hors périmètre"
        print(f"{best_score:.3f} — {scope} — {item.q}")


TF-IDF
0.478 — avec source — Combien de temps de préavis pour un contrat mensuel ?
0.380 — avec source — Quels frais si je quitte mon abonnement annuel avant la fin ?
0.174 — avec source — Je déménage, puis-je partir sans payer ?
0.170 — avec source — Ma ligne est coupée, que faire ?
0.314 — avec source — Puis-je proposer une remise à un client qui veut partir ?
0.452 — hors périmètre — Quel est le cours de l'action de l'entreprise ?

BM25
5.453 — avec source — Combien de temps de préavis pour un contrat mensuel ?
4.334 — avec source — Quels frais si je quitte mon abonnement annuel avant la fin ?
0.759 — avec source — Je déménage, puis-je partir sans payer ?
1.618 — avec source — Ma ligne est coupée, que faire ?
2.835 — avec source — Puis-je proposer une remise à un client qui veut partir ?
2.082 — hors périmètre — Quel est le cours de l'action de l'entreprise ?

Embeddings
0.634 — avec source — Combien de temps de préavis pour un contrat mensuel ?
0.552 — avec source — Quels frais si

## 2. Appliquer une règle d’acceptation ou de refus

Chaque méthode possède un seuil distinct. La règle refuse une recherche sans résultat,
un score nul ou un meilleur score inférieur au seuil. Un score positif égal ou
supérieur au seuil conduit à l’acceptation.

Les sources restent disponibles pour examiner les décisions. Une acceptation ne
prouve pas que le contexte contient la réponse et ne valide pas individuellement
tous les documents retournés.


In [4]:
# Valeurs illustratives chargées depuis configs/retrieval.yaml.
thresholds = settings.retrieval.abstention_thresholds
policies = {name: AbstentionPolicy(threshold=value) for name, value in thresholds.items()}

reasons = {
    "no_results": "Aucun document n’a été retrouvé.",
    "no_positive_score": "Aucun document ne possède un score positif.",
    "below_threshold": "Le meilleur score est inférieur au seuil requis.",
    "threshold_met": "Le meilleur score atteint le seuil requis.",
}

question = "Quels frais pour une résiliation annuelle ?"
print("QUESTION")
print(question)

for number, (name, retriever) in enumerate(retrievers.items(), start=1):
    results = retriever.retrieve(question, top_k=settings.retrieval.top_k)
    decision = policies[name].decide(results)
    status = "ACCEPTÉE" if decision.accepted else "REFUSÉE"
    best_score = (
        f"{decision.best_score:.3f}" if decision.best_score is not None else "Non disponible"
    )

    print("\n" + "=" * 70)
    print(f"{number}. MÉTHODE : {name}")
    print("-" * 70)
    print(f"Décision         : {status}")
    print(f"Explication      : {reasons[decision.reason]}")
    print()
    print(f"Meilleur score   : {best_score}")
    print(f"Seuil appliqué   : {decision.threshold:.3f}")
    print("\nSources retrouvées :")

    if not decision.results:
        print("  Aucun résultat.")
    for rank, result in enumerate(decision.results, start=1):
        print(f"  {rank}. {result.document.id} — score : {result.score:.3f}")
    print("=" * 70)

print(
    "\nUne acceptation indique que le seuil est franchi ; elle ne garantit pas que la réponse est disponible."
)

QUESTION
Quels frais pour une résiliation annuelle ?

1. MÉTHODE : TF-IDF
----------------------------------------------------------------------
Décision         : REFUSÉE
Explication      : Le meilleur score est inférieur au seuil requis.

Meilleur score   : 0.212
Seuil appliqué   : 0.300

Sources retrouvées :
  1. D2 — score : 0.212
  2. D5 — score : 0.123
  3. D3 — score : 0.077

2. MÉTHODE : BM25
----------------------------------------------------------------------
Décision         : ACCEPTÉE
Explication      : Le meilleur score atteint le seuil requis.

Meilleur score   : 2.255
Seuil appliqué   : 2.000

Sources retrouvées :
  1. D2 — score : 2.255
  2. D5 — score : 1.418
  3. D3 — score : 0.714

3. MÉTHODE : Embeddings
----------------------------------------------------------------------
Décision         : ACCEPTÉE
Explication      : Le meilleur score atteint le seuil requis.

Meilleur score   : 0.536
Seuil appliqué   : 0.400

Sources retrouvées :
  1. D2 — score : 0.536
  2. D5

## 3. Mesurer les erreurs de décision

Le périmètre attendu est déduit des annotations uniquement lors de l’évaluation.
La politique d’abstention prend sa décision sans consulter les sources attendues.

- **Taux de faux refus** : questions avec source refusées / questions avec source.
- **Taux de fausses acceptations** : hors périmètre acceptés / questions hors périmètre.
- **Taux d’acceptation** : toutes les questions acceptées / toutes les questions.

Un taux vaut `None` lorsque le groupe correspondant est absent. Le taux d’acceptation
ne mesure ni la justesse des sources ni celle d’une future réponse.


In [5]:
evaluator = AbstentionEvaluator()
reports = {
    name: evaluator.evaluate(
        retriever, dataset.questions, policies[name], top_k=settings.retrieval.top_k
    )
    for name, retriever in retrievers.items()
}

for name, report in reports.items():
    print(f"\n{name} — seuil {report.threshold}")
    print(f"Faux refus : {report.false_refusals}/{report.in_scope_count}")
    print(f"Fausses acceptations : {report.false_acceptances}/{report.out_of_scope_count}")
    print("Taux de faux refus :", report.false_refusal_rate)
    print("Taux de fausses acceptations :", report.false_acceptance_rate)
    print("Taux d’acceptation :", report.acceptance_rate)


TF-IDF — seuil 0.3
Faux refus : 2/5
Fausses acceptations : 1/1
Taux de faux refus : 0.4
Taux de fausses acceptations : 1.0
Taux d’acceptation : 0.6666666666666666

BM25 — seuil 2.0
Faux refus : 2/5
Fausses acceptations : 1/1
Taux de faux refus : 0.4
Taux de fausses acceptations : 1.0
Taux d’acceptation : 0.6666666666666666

Embeddings — seuil 0.4
Faux refus : 1/5
Fausses acceptations : 0/1
Taux de faux refus : 0.2
Taux de fausses acceptations : 0.0
Taux d’acceptation : 0.6666666666666666


## 4. Examiner les décisions et leurs erreurs

Le détail par question permet de distinguer un faux refus d’une fausse acceptation.
Pour les questions acceptées, la présence des sources attendues est également
vérifiée : une politique peut accepter une question alors que le retrieval a échoué.


In [6]:
for name, report in reports.items():
    print(f"\n{name}")
    for case in report.cases:
        expected_in_scope = bool(case.question.gold_docs)
        print(case.question.q)
        print("Attendu :", case.question.gold_docs or "Hors périmètre")
        print("Accepté :", case.decision.accepted, "—", case.decision.reason)
        print("Erreur de périmètre :", case.decision.accepted != expected_in_scope)
        retrieved_ids = [result.document.id for result in case.decision.results]
        print("Retrouvé :", retrieved_ids)
        if expected_in_scope and case.decision.accepted:
            print(
                "Source attendue présente :",
                bool(set(retrieved_ids) & set(case.question.gold_docs)),
            )
        print()


TF-IDF
Combien de temps de préavis pour un contrat mensuel ?
Attendu : ['D1']
Accepté : True — threshold_met
Erreur de périmètre : False
Retrouvé : ['D1', 'D4', 'D3']
Source attendue présente : True

Quels frais si je quitte mon abonnement annuel avant la fin ?
Attendu : ['D2']
Accepté : True — threshold_met
Erreur de périmètre : False
Retrouvé : ['D2', 'D4', 'D3']
Source attendue présente : True

Je déménage, puis-je partir sans payer ?
Attendu : ['D3']
Accepté : False — below_threshold
Erreur de périmètre : True
Retrouvé : ['D3', 'D6', 'D5']

Ma ligne est coupée, que faire ?
Attendu : ['D4']
Accepté : False — below_threshold
Erreur de périmètre : True
Retrouvé : ['D1', 'D4']

Puis-je proposer une remise à un client qui veut partir ?
Attendu : ['D5']
Accepté : True — threshold_met
Erreur de périmètre : False
Retrouvé : ['D1', 'D2', 'D3']
Source attendue présente : False

Quel est le cours de l'action de l'entreprise ?
Attendu : Hors périmètre
Accepté : True — threshold_met
Erreur de 

## 5. Explorer le compromis entre les seuils

Les grilles montrent comment les décisions évoluent lorsque le seuil augmente.
Un seuil plus exigeant peut réduire les fausses acceptations tout en augmentant
les faux refus. Avec un seul exemple hors périmètre, son taux d’erreur prend uniquement
les valeurs zéro ou un : cette exploration ne remplace pas une validation sur des
exemples plus nombreux et indépendants.


In [7]:
threshold_grids = settings.evaluation.threshold_grids
rows = [
    "| Méthode | Seuil illustratif | Faux refus | Fausses acceptations | Taux d’acceptation |",
    "|---|---:|---:|---:|---:|",
]
for name, values in threshold_grids.items():
    for threshold in values:
        report = evaluator.evaluate(
            retrievers[name],
            dataset.questions,
            AbstentionPolicy(threshold=threshold),
            top_k=settings.retrieval.top_k,
        )
        rows.append(
            f"| {name} | {threshold:.2f} | {report.false_refusals}/{report.in_scope_count} | "
            f"{report.false_acceptances}/{report.out_of_scope_count} | {report.acceptance_rate:.3f} |"
        )
display(Markdown("\n".join(rows)))

| Méthode | Seuil illustratif | Faux refus | Fausses acceptations | Taux d’acceptation |
|---|---:|---:|---:|---:|
| TF-IDF | 0.00 | 0/5 | 1/1 | 1.000 |
| TF-IDF | 0.10 | 0/5 | 1/1 | 1.000 |
| TF-IDF | 0.30 | 2/5 | 1/1 | 0.667 |
| TF-IDF | 0.50 | 5/5 | 0/1 | 0.000 |
| TF-IDF | 0.70 | 5/5 | 0/1 | 0.000 |
| BM25 | 0.00 | 0/5 | 1/1 | 1.000 |
| BM25 | 0.50 | 0/5 | 1/1 | 1.000 |
| BM25 | 1.00 | 1/5 | 1/1 | 0.833 |
| BM25 | 2.00 | 2/5 | 1/1 | 0.667 |
| BM25 | 4.00 | 3/5 | 0/1 | 0.333 |
| Embeddings | 0.00 | 0/5 | 1/1 | 1.000 |
| Embeddings | 0.20 | 0/5 | 1/1 | 1.000 |
| Embeddings | 0.40 | 1/5 | 0/1 | 0.667 |
| Embeddings | 0.60 | 4/5 | 0/1 | 0.167 |
| Embeddings | 0.80 | 5/5 | 0/1 | 0.000 |

## Prolongement de l’analyse

Le [notebook sur le prompt et la sécurité](05_prompt_and_safety.ipynb) examine comment
la décision d’abstention intervient avant la préparation du contexte et comment une
injection documentaire connue est bloquée.
